# Chương 2 – Cơ bản về Xử lý ảnh số
## Bài tập thực hành

Mục tiêu:
- Biểu diễn ảnh dưới dạng ma trận số và thao tác trực tiếp trên ma trận đó.
- Thực hành khái niệm **lấy mẫu (sampling)** và **lượng tử hóa (quantization)** — gắn với độ phân giải
  không gian và độ phân giải mức xám đã học trong slide Chương 2.
- Thực hành các phép chuyển đổi **không gian màu** và xây dựng **histogram** thủ công.

**Dữ liệu ảnh**: thư mục `images/` (gồm `HoGuom.jpg`, `gao3.png`) và `images/pieces_yellow/` (bài tập nâng cao).

In [ ]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

%matplotlib inline

IMG_DIR = 'images'

## 1. Biểu diễn ảnh dưới dạng ma trận số

Một ảnh màu là một ma trận 3 chiều `(height, width, 3)`. Mỗi điểm ảnh (pixel) là một bộ ba giá trị
cường độ (B, G, R theo cách lưu của OpenCV).

In [ ]:
img = cv2.imread(os.path.join(IMG_DIR, 'HoGuom.jpg'))
print("Shape:", img.shape)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

**Yêu cầu 1**: In ra 3 giá trị (B, G, R) tại điểm ảnh có toạ độ `(row=50, col=100)`.

**Yêu cầu 2**: Tạo một bản sao của ảnh, gán một vùng ảnh 60x60 pixel ở góc trên-trái (từ (0,0) đến (60,60)) thành màu đỏ thuần tuý (B=0, G=0, R=255), rồi hiển thị kết quả để thấy rằng ảnh chỉ là một ma trận số có thể thao tác trực tiếp.

In [ ]:
pixel_value = img[50, 100]
print("Giá trị (B, G, R) tại (50, 100):", pixel_value)

img_copy = img.copy()
img_copy[0:60, 0:60] = (0, 0, 255)

plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
plt.show()

## 2. Lấy mẫu ảnh (Sampling) và độ phân giải không gian

Độ phân giải không gian của ảnh phụ thuộc vào số lượng điểm ảnh dùng để biểu diễn ảnh. Ta sẽ mô phỏng việc
giảm độ phân giải không gian bằng cách **thu nhỏ ảnh** (lấy mẫu thưa hơn) rồi **phóng to lại** về kích thước
ban đầu, để quan sát sự mất mát chi tiết.

**Yêu cầu**: viết hàm `resample(img, factor)`:
- Thu nhỏ ảnh xuống còn `1/factor` kích thước ban đầu bằng `cv2.resize`.
- Phóng to ảnh đã thu nhỏ về đúng kích thước ban đầu (dùng `interpolation=cv2.INTER_NEAREST` để thấy rõ
  hiệu ứng "ô vuông" của việc lấy mẫu thưa).

Áp dụng hàm với `factor = 1 (giữ nguyên), 4, 16, 64` và hiển thị 4 ảnh kết quả trên 1 hàng.

In [ ]:
def resample(img, factor):
    h, w = img.shape[:2]
    new_w = max(1, w // factor)
    new_h = max(1, h // factor)
    small = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    restored = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    return restored

gray = cv2.imread(os.path.join(IMG_DIR, 'HoGuom.jpg'), cv2.IMREAD_GRAYSCALE)
factors = [1, 4, 16, 64]

plt.figure(figsize=(4 * len(factors), 4))
for i, f in enumerate(factors):
    result = resample(gray, f)
    plt.subplot(1, len(factors), i + 1)
    plt.imshow(result, cmap='gray')
    plt.title(f'factor = {f}')
    plt.axis('off')
plt.show()

## 3. Lượng tử hóa ảnh (Quantization) và độ phân giải mức xám

Độ phân giải mức xám (intensity resolution) là số mức cường độ sáng khác nhau dùng để biểu diễn ảnh
(ví dụ 256, 128, 64, ..., 2 mức xám — tương ứng 8, 7, 6, ..., 1 bit/pixel).

**Yêu cầu**: viết hàm `quantize(img, levels)` chia đều dải giá trị [0, 255] thành số lượng `levels` mức,
rồi làm tròn mỗi giá trị pixel về mức gần nhất. Áp dụng cho ảnh `gao3.png` với `levels = 256, 32, 8, 4, 2`.

In [ ]:
def quantize(img, levels):
    if not 2 <= levels <= 256:
        raise ValueError('levels phải nằm trong [2, 256]')
    step = 255.0 / (levels - 1)
    result = np.rint(img.astype(np.float64) / step) * step
    result = np.clip(np.rint(result), 0, 255).astype(np.uint8)
    return result

rice = cv2.imread(os.path.join(IMG_DIR, 'gao3.png'), cv2.IMREAD_GRAYSCALE)
level_list = [256, 32, 8, 4, 2]

plt.figure(figsize=(4 * len(level_list), 4))
for i, lv in enumerate(level_list):
    q = quantize(rice, lv)
    plt.subplot(1, len(level_list), i + 1)
    plt.imshow(q, cmap='gray', vmin=0, vmax=255)
    plt.title(f'{lv} mức xám')
    plt.axis('off')
plt.show()

## 4. Không gian màu (Color spaces)

OpenCV hỗ trợ chuyển đổi giữa nhiều không gian màu bằng `cv2.cvtColor`: `COLOR_BGR2GRAY`,
`COLOR_BGR2HSV`, `COLOR_BGR2Lab`, ...

**Yêu cầu**: chuyển ảnh `HoGuom.jpg` sang 3 không gian màu GRAY, HSV, Lab và hiển thị cả 3 kết quả
(cùng với ảnh RGB gốc) trên 1 hàng gồm 4 ảnh.

In [ ]:
img = cv2.imread(os.path.join(IMG_DIR, 'HoGuom.jpg'))

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)

titles = ['RGB', 'GRAY', 'HSV', 'Lab']
images = [img_rgb, img_gray, img_hsv, img_lab]
plt.figure(figsize=(16, 4))
for i, (im, t) in enumerate(zip(images, titles)):
    plt.subplot(1, 4, i + 1)
    plt.imshow(im, cmap='gray' if t == 'GRAY' else None)
    plt.title(t)
    plt.axis('off')
plt.show()

### Hiển thị riêng từng kênh màu B, G, R

**Yêu cầu**: với ảnh `HoGuom.jpg`, hiển thị lần lượt từng kênh màu B, G, R bằng cách gán 2 kênh còn lại
bằng 0 (giữ nguyên 3 chiều để `imshow` hiển thị đúng màu của kênh đó).

In [ ]:
b_channel = img.copy()
g_channel = img.copy()
r_channel = img.copy()

b_channel[:, :, 1:] = 0
g_channel[:, :, 0] = 0
g_channel[:, :, 2] = 0
r_channel[:, :, :2] = 0

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.title('Kênh B'); plt.imshow(cv2.cvtColor(b_channel, cv2.COLOR_BGR2RGB))
plt.subplot(1, 3, 2); plt.title('Kênh G'); plt.imshow(cv2.cvtColor(g_channel, cv2.COLOR_BGR2RGB))
plt.subplot(1, 3, 3); plt.title('Kênh R'); plt.imshow(cv2.cvtColor(r_channel, cv2.COLOR_BGR2RGB))
plt.show()

## 5. Xây dựng histogram thủ công

Histogram (biểu đồ tần suất mức xám) mô tả số lượng điểm ảnh ứng với mỗi mức cường độ sáng — liên hệ trực
tiếp đến khái niệm độ phân giải mức xám đã học ở trên.

**Yêu cầu**: viết hàm `compute_histogram(gray_img)` KHÔNG dùng `cv2.calcHist`/`plt.hist`/`np.histogram`
có sẵn, mà tự tính bằng vòng lặp: trả về 1 vector `numpy` gồm 256 phần tử, phần tử thứ `i` là số lượng
điểm ảnh có giá trị bằng `i`. Sau đó vẽ kết quả bằng `plt.plot` và so sánh với `plt.hist` để kiểm tra.

In [ ]:
def compute_histogram(gray_img):
    histogram = np.zeros((256,))
    for value in gray_img.ravel():
        histogram[int(value)] += 1
    return histogram

rice = cv2.imread(os.path.join(IMG_DIR, 'gao3.png'), cv2.IMREAD_GRAYSCALE)
hist = compute_histogram(rice)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.title('Tự tính bằng vòng lặp'); plt.plot(hist)
plt.subplot(1, 2, 2); plt.title('Kiểm tra bằng plt.hist'); plt.hist(rice.ravel(), bins=256)
plt.show()

## 6. Bài tập nâng cao: Không gian màu và điều kiện chiếu sáng

Thư mục `images/pieces_yellow/` chứa các mảnh ảnh nhỏ (trích ra từ ảnh khối rubik) có cùng màu vàng nhưng
được chụp ở các điều kiện chiếu sáng khác nhau.

**Yêu cầu**: đọc toàn bộ ảnh trong thư mục, vẽ biểu đồ phân bố 2 chiều (`plt.hist2d`) của các cặp kênh
(B,G), (B,R), (R,G) trong không gian màu RGB. Từ đó nhận xét: khi ánh sáng thay đổi, các kênh màu trong
RGB thay đổi như thế nào? (Gợi ý mở rộng: thử lại với không gian màu HSV hoặc YCrCb và so sánh kênh nào
ổn định hơn khi ánh sáng thay đổi — có thể dùng `cv2.cvtColor`).

In [ ]:
files = sorted(glob.glob(os.path.join(IMG_DIR, 'pieces_yellow', '*.jpg')))
print("Số ảnh:", len(files))

B, G, R = np.array([]), np.array([]), np.array([])

channels = []
for path in files:
    piece = cv2.imread(path)
    if piece is None:
        raise FileNotFoundError(path)
    channels.append(piece.reshape(-1, 3))

pixels = np.concatenate(channels, axis=0)
B, G, R = pixels[:, 0], pixels[:, 1], pixels[:, 2]

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1); plt.hist2d(B, G, bins=10, norm=LogNorm()); plt.xlabel('B'); plt.ylabel('G'); plt.colorbar()
plt.subplot(1, 3, 2); plt.hist2d(B, R, bins=10, norm=LogNorm()); plt.xlabel('B'); plt.ylabel('R'); plt.colorbar()
plt.subplot(1, 3, 3); plt.hist2d(R, G, bins=10, norm=LogNorm()); plt.xlabel('R'); plt.ylabel('G'); plt.colorbar()
plt.show()

# Nhận xét: khi độ chiếu sáng thay đổi, cả B, G và R cùng dịch chuyển và có
# tương quan mạnh, nên màu trong RGB không ổn định theo độ sáng. Trong HSV, H
# thường ổn định hơn V vì H mô tả sắc màu còn V chứa phần lớn thông tin độ sáng.